# Lesson 2.2: loc, iloc, Filtering aur Sorting

Real analysis ke sawal:
- "Sirf **Babar** ki innings dikhao."
- "**India** ke khilaf **50+** scores kaun se hain?"
- "**Top 3** innings kaunsi hain?"
- "Jahan runs **0** hain, wahan `boundary_pct` ko **0** kar do."

NumPy mein mask se filtering ki thi (`runs[runs >= 50]`). Pandas mein bilkul wahi idea, bas ab **poori rows** milti hain.

| Kaam | SQL mein |
|---|---|
| Filtering | `WHERE` |
| Columns chunna | `SELECT col1, col2` |
| Sorting | `ORDER BY ... LIMIT` |
| `loc` se value badalna | `UPDATE ... SET ... WHERE` |

In [ ]:
import pandas as pd

data = {
    "player":           ["Babar", "Rizwan", "Babar", "Fakhar", "Rizwan", "Babar", "Fakhar", "Rizwan"],
    "opposition":       ["India", "Australia", "England", "India", "England", "Australia", "England", "India"],
    "batting_position": [3, 1, 3, 2, 1, 3, 2, 1],
    "balls":            [30, 50, 15, 60, 28, 3, 70, 40],
    "fours":            [5, 8, 1, 10, 3, 0, 12, 4],
    "sixes":            [1, 2, 0, 4, 1, 0, 3, 2],
    "runs":             [45, 67, 12, 89, 34, 0, 102, 51],
}

df = pd.DataFrame(data)
df

Left side pe **0 se 7** numbers = **index**: har row ka label/ID, jaise MySQL ka primary key.

Columns ke **positions** (iloc ke liye kaam aayenge):
```
0 player | 1 opposition | 2 batting_position | 3 balls | 4 fours | 5 sixes | 6 runs
```

⚠️ Har command **alag cell** mein, kyunki notebook sirf cell ki **aakhri line** khud dikhati hai. (Purane notebook mein kai lines ek cell mein thin, isliye sirf aakhri ka output dikhta tha.)

---
## Part 1: `iloc` — position se (NumPy jaisa)
`iloc[rows, columns]`, dono **numbers** se. Slice ka end **shamil nahi** (Python jaisa).

In [ ]:
df.iloc[0]          # pehli row (Series ki shakal mein)

In [ ]:
df.iloc[0:3]        # rows 0, 1, 2  → 3 rows

In [ ]:
df.iloc[2, 6]       # row 2, column 6 (runs) → 12

In [ ]:
df.iloc[:, 0:2]     # saari rows, pehle 2 columns (player, opposition)

In [ ]:
df.iloc[-3:]        # aakhri 3 rows (negative index peeche se ginta hai)

---
## Part 2: `loc` — label/naam se
`loc[rows, columns]`: rows **index label** se, columns **naam** se. Slice ka end **shamil hai**.

In [ ]:
df.loc[0, "runs"]                      # row 0 ke runs → 45

In [ ]:
df.loc[0:3, ["player", "runs"]]        # rows 0 SE 3 (4 rows!), do columns

### `iloc` vs `loc`

| | `iloc` | `loc` |
|---|---|---|
| Kis se dhoondta hai | **position** (0, 1, 2…) | **label** (index ka naam, column ka naam) |
| Columns | number: `iloc[:, 0:2]` | naam: `loc[:, ["player", "runs"]]` |
| Slice `0:3` | **3 rows** (end shamil nahi) | **4 rows** (end shamil) |

**Practical rule:** ML mein **90% waqt `loc`** aur seedha filtering. Column **naam** se kaam saaf aur mehfooz hai. `iloc` tab, jab "pehli 5 / aakhri 3 rows" jaisi position chahiye.

In [ ]:
# Farq khud dekhein
print("iloc[0:3]:", len(df.iloc[0:3]), "rows")
print("loc[0:3]: ", len(df.loc[0:3]), "rows")

---
## Part 3: Filtering

`df[df["runs"] >= 50]` do hisson mein:
```
df["runs"] >= 50     →  har row ke liye True/False (mask), NumPy jaisa
df[ mask ]           →  sirf True wali rows rakho
```

In [ ]:
df["runs"] >= 50      # pehle mask khud dekhein

In [ ]:
df[df["runs"] >= 50]

In [ ]:
df[df["player"] == "Babar"]

### Do conditions
NumPy wala hi rule: `&` (and), `|` (or), aur **har condition brackets mein**.

| ❌ Ghalat | ✅ Sahi |
|---|---|
| `df["opposition"] == "India" and df["runs"] >= 50` | `(df["opposition"] == "India") & (df["runs"] >= 50)` |

In [ ]:
df[(df["opposition"] == "India") & (df["runs"] >= 50)]

### `isin`: "inmein se koi bhi"
`(opp == "India") | (opp == "Australia")` ka chhota tareeqa. SQL: `WHERE opposition IN ('India', 'Australia')`

In [ ]:
df[df["opposition"].isin(["India", "Australia"])]

### Filter + columns ek saath: `df.loc[mask, columns]`
SQL: `SELECT player, opposition, runs FROM df WHERE runs >= 50`

In [ ]:
df.loc[df["runs"] >= 50, ["player", "opposition", "runs"]]

---
## Part 4: Sorting
- `sort_values("runs", ascending=False)` → zyada se kam
- `.head(3)` → **top 3**. SQL: `ORDER BY runs DESC LIMIT 3`

In [ ]:
df.sort_values("runs", ascending=False)

In [ ]:
df.sort_values("runs", ascending=False).head(3)

In [ ]:
# Shortcut: nlargest / nsmallest
df.nlargest(3, "runs")[["player", "runs"]]

---
## Part 5: `loc` se values badalna (bohat important)

Lesson 2.1 mein `boundary_pct` banaya tha. Babar ki **duck** (runs 0, boundary 0) pe `0 / 0` = **NaN** aaya.

In [ ]:
df["boundary_runs"] = df["fours"] * 4 + df["sixes"] * 6
df["boundary_pct"] = df["boundary_runs"] / df["runs"] * 100

print("NaN pehle:", df["boundary_pct"].isnull().sum())   # 1
df.loc[5]   # duck wali row

**Faisla (cricket ki soch se):** "0 runs mein 0% boundaries se aaye" → 0 kar do.

```python
df.loc[df["runs"] == 0, "boundary_pct"] = 0
```
= **"Jin rows mein runs 0 hain, unke `boundary_pct` mein 0 likh do."**
SQL: `UPDATE df SET boundary_pct = 0 WHERE runs = 0`

In [ ]:
df.loc[df["runs"] == 0, "boundary_pct"] = 0
print("NaN baad:", df["boundary_pct"].isnull().sum())    # 0

⚠️ **Ye ghalat tareeqa kabhi na likhein:**
```python
df[df["runs"] == 0]["boundary_pct"] = 0    # ❌
```
Ye pehle filter ki ek **copy** banata hai aur copy ko badalta hai, asal `df` ko nahi. Pandas 3.0 mein ye **kuch nahi badlega**.
> Values badalni hon to hamesha **`df.loc[condition, column] = value`**.

(Har NaN ka faisla 0 nahi hota. Ye yahan cricket ki mantiq se hamara faisla tha.)

---
## Part 6: Exercise ke jawab (har sawal alag cell)

### 1. Sirf Rizwan ki innings: player, opposition, runs

In [ ]:
df.loc[df["player"] == "Rizwan", ["player", "opposition", "runs"]]

### 2. England ke khilaf 40+ runs wali innings

In [ ]:
df[(df["opposition"] == "England") & (df["runs"] > 40)]

Sirf Fakhar ki 102.

### 3. Openers (position 1) ne kitni innings mein 50+ banaye? Sirf ginti
Do tareeqe. Doosra NumPy wala trick: **True = 1**, to `.sum()` = ginti.

In [ ]:
len(df[(df["batting_position"] == 1) & (df["runs"] >= 50)])

In [ ]:
((df["batting_position"] == 1) & (df["runs"] >= 50)).sum()

**2** (Rizwan ke 67 aur 51).

### 4. Sab se kam runs wali 3 innings

In [ ]:
df.sort_values("runs").head(3)[["player", "runs"]]

In [ ]:
df.nsmallest(3, "runs")[["player", "runs"]]   # same, chhota tareeqa

### 5. iloc se aakhri 3 rows, pehle 3 columns

In [ ]:
df.iloc[-3:, 0:3]

### 6. `fifty_plus` column (True/False), phir sirf Babar

In [ ]:
df["fifty_plus"] = df["runs"] >= 50
df.loc[df["player"] == "Babar", ["player", "runs", "fifty_plus"]]

Babar ki teeno innings **False** (45, 12, 0).

---
## Khulasa
| Kaam | Code |
|---|---|
| Position se | `df.iloc[rows, cols]` (end shamil nahi) |
| Naam se | `df.loc[rows, cols]` (end shamil) |
| Filter | `df[df["col"] >= x]` |
| Do conditions | `(cond1) & (cond2)`, `(cond1) \| (cond2)` |
| Kai values mein se | `df["col"].isin([...])` |
| Filter + columns | `df.loc[mask, ["a", "b"]]` |
| Sort / top N | `sort_values(..., ascending=False).head(n)`, `nlargest` |
| Value badalna | `df.loc[mask, "col"] = value` |
| Ginti | `mask.sum()` |